# 🔍 Inference & Visualization
## Glaucoma Detection — Trained Model

Load the best checkpoint and:
- Run inference on individual images
- Visualize Grad-CAM attention maps
- Explore prediction confidence distributions

---

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
from omegaconf import OmegaConf

sys.path.insert(0, '../src')
from model import build_model
from utils import set_seed, load_checkpoint
from dataset import get_transforms
from gradcam import GradCAM, overlay_heatmap

# Config
cfg    = OmegaConf.load('../configs/efficientnet_b3.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
set_seed(cfg.experiment.seed)
print(f'Device: {device}')

## 1. Load Model

In [ ]:
model = build_model(
    model_name   = cfg.model.architecture,
    pretrained   = False,
    dropout_rate = cfg.model.dropout_rate,
    device       = str(device),
)

CHECKPOINT = '../outputs/checkpoints/best_model.pth'
epoch, metrics = load_checkpoint(CHECKPOINT, model, device=str(device))
model.eval()

print(f'Loaded checkpoint from epoch {epoch}')
print('Metrics:', metrics)

## 2. Single Image Inference

In [ ]:
def predict_image(image_path, true_label=None):
    """Predict GON label for a single image and show Grad-CAM."""
    img_rgb = np.array(Image.open(image_path).convert('RGB'))
    tf      = get_transforms('test', cfg.dataset.image_size)
    tensor  = tf(image=img_rgb)['image'].unsqueeze(0).to(device)

    # Grad-CAM
    gradcam = GradCAM(model)
    heatmap = gradcam(tensor)
    gradcam.remove_hooks()

    prob    = torch.sigmoid(model(tensor)).item()
    pred    = 'GON+' if prob >= 0.5 else 'GON−'
    overlay = overlay_heatmap(img_rgb, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original Image', fontsize=12)
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM Heatmap', fontsize=12)
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay', fontsize=12)

    for ax in axes: ax.axis('off')

    color = '#EF4444' if pred == 'GON+' else '#10B981'
    title = f'Prediction: {pred}  (p = {prob:.3f})'
    if true_label:
        match = '✓' if pred.replace('−', '-') == true_label else '✗'
        title += f'  |  Ground Truth: {true_label}  {match}'
    fig.suptitle(title, fontsize=14, fontweight='bold', color=color)
    plt.tight_layout()
    plt.show()

# Example — replace with actual image path
# predict_image('../data/HYDR/Images/188_1.jpg', true_label='GON+')

## 3. Batch Inference on Test Set

In [ ]:
from dataset import build_dataloaders
from utils import compute_metrics
from sklearn.metrics import classification_report

_, _, test_loader, _ = build_dataloaders(
    labels_csv        = cfg.paths.labels_csv,
    images_dir        = cfg.paths.images_dir,
    image_size        = cfg.dataset.image_size,
    batch_size        = cfg.training.batch_size,
    num_workers       = 0,
    min_quality_score = cfg.dataset.min_quality_score,
    val_size          = cfg.dataset.val_size,
    test_size         = cfg.dataset.test_size,
    seed              = cfg.experiment.seed,
)

all_labels, all_probs = [], []
with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        probs  = torch.sigmoid(model(images)).squeeze(1).cpu().numpy()
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

labels = np.array(all_labels)
probs  = np.array(all_probs)
preds  = (probs >= 0.5).astype(int)

metrics = compute_metrics(labels, probs)
print('=== Test Set Metrics ===')
for k, v in metrics.items():
    print(f'  {k:15s}: {v}')
print()
print(classification_report(labels, preds, target_names=['GON-', 'GON+']))

## 4. Prediction Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability histogram by true class
axes[0].hist(probs[labels==0], bins=30, alpha=0.65, color='#10B981', label='GON− (true)', edgecolor='white')
axes[0].hist(probs[labels==1], bins=30, alpha=0.65, color='#EF4444', label='GON+ (true)', edgecolor='white')
axes[0].axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold')
axes[0].set_xlabel('P(GON+)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Prediction Probability by True Class', fontsize=13, fontweight='bold')
axes[0].legend()

# Confidence of correct vs incorrect predictions
correct_probs   = probs[preds == labels]
incorrect_probs = probs[preds != labels]
axes[1].hist(correct_probs,   bins=20, alpha=0.7, color='#3B82F6', label=f'Correct   (n={len(correct_probs)})',   edgecolor='white')
axes[1].hist(incorrect_probs, bins=20, alpha=0.7, color='#F97316', label=f'Incorrect (n={len(incorrect_probs)})', edgecolor='white')
axes[1].set_xlabel('Predicted Probability', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Confidence: Correct vs Incorrect', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/inference_confidence.png', dpi=150)
plt.show()